In [1]:
import pandas as pd

In [ ]:
df_occupations_with_sectors = pd.read_csv("occupations_with_sectors.csv")
df_other_services = df_occupations_with_sectors[df_occupations_with_sectors['SUBSECTOR']=="Other services"]
df_other_services.to_csv("occupations_with_sectors_others.csv",index=False)

In [ ]:
#mapp sector_id to occupation_with_sector
import json
import pandas as pd
sector_ids = None
sub_sector_ids = None
with open("sector_sub_sector_ids.json","r") as file:
    ids = json.load(file)
    sector_ids = ids['sectors']
    sub_sector_ids = ids['sub_sectors']

sectors_id_map = {sector['name']:sector['id'] for sector in sector_ids}
sub_sector_id_map = {sub_sector['name']:sub_sector['id'] for sub_sector in sub_sector_ids}
df_occupations_with_sectors = pd.read_csv("occupations_with_sectors.csv")
# df_occupations_with_sectors[]
df_occupations_with_sectors['SECTORID'] = df_occupations_with_sectors['SECTOR'].map(sectors_id_map)
df_occupations_with_sectors['SUBSECTORID'] = df_occupations_with_sectors['SUBSECTOR'].map(sub_sector_id_map)
# df_occupations_with_sectors.head()
df_occupations_with_sectors.to_csv("occupations_with_sectors.csv",index=False)


In [19]:
#merge occupation and skills and keep only important files
import pandas as pd


def rename_columns(df, prefix):
    """Rename standard metadata columns with a prefix."""
    return df.rename(columns={
        "PREFERREDLABEL": f"{prefix}_PREFERREDLABEL",
        "ALTLABELS": f"{prefix}_ALTLABELS",
        "DESCRIPTION": f"{prefix}_DESCRIPTION"
    })


def validate_columns(df, required_cols, name):
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")


def combine_occupation_skill(
    occupation_skill_path,
    skills_path,
    occupation_path,
    output_path=None
):
    # Load CSVs
    df_relation = pd.read_csv(occupation_skill_path, dtype=str)
    df_skills = pd.read_csv(skills_path, dtype=str)
    df_occupation = pd.read_csv(occupation_path, dtype=str)

    # Validate
    validate_columns(df_relation, ["OCCUPATIONID", "SKILLID"], "relation file")
    validate_columns(df_skills, ["ID"], "skills file")
    validate_columns(df_occupation, ["ID"], "occupation file")

    # Rename
    df_skills = rename_columns(df_skills, "SKILL")
    df_occupation = rename_columns(df_occupation, "OCCUPATION")

    # Merge occupation
    df_merge = df_relation.merge(
        df_occupation[
            [
                "ID",
                "OCCUPATION_PREFERREDLABEL",
                "OCCUPATION_ALTLABELS",
                "OCCUPATION_DESCRIPTION",
                "SECTOR",
                "SECTORID",
                "SUBSECTOR",
                "SUBSECTORID"
            ]
        ],
        how="left",
        left_on="OCCUPATIONID",
        right_on="ID"
    ).drop(columns=["ID"])

    # Merge skills
    df_merge = df_merge.merge(
        df_skills[
            [
                "ID",
                "SKILL_PREFERREDLABEL",
                "SKILL_ALTLABELS",
                "SKILL_DESCRIPTION"
            ]
        ],
        how="left",
        left_on="SKILLID",
        right_on="ID"
    ).drop(columns=["ID"])

    # Final order
    column_order = [
        "OCCUPATIONID",
        "OCCUPATION_PREFERREDLABEL",
        "OCCUPATION_ALTLABELS",
        "OCCUPATION_DESCRIPTION",
        "SECTOR",
        "SECTORID",
        "SUBSECTOR",
        "SUBSECTORID",
        "SKILLID",
        "SKILL_PREFERREDLABEL",
        "SKILL_ALTLABELS",
        "SKILL_DESCRIPTION",
        "RELATIONTYPE"
    ]

    df_merge = df_merge[column_order]

    # Optional save
    if output_path:
        df_merge.to_csv(output_path, index=False)

    return df_merge


# Usage
df = combine_occupation_skill(
    "occupation_to_skill_relations.csv",
    "skills.csv",
    "occupations_with_sectors.csv",
    output_path="combined_occupation_to_skill.csv"
)

print(df.head())

               OCCUPATIONID      OCCUPATION_PREFERREDLABEL  \
0  6893387fba407998dac8dd2b              musical conductor   
1  6893387fba407998dac8dd29                 music director   
2  6893387fba407998dac8dd2c      choirmaster/choirmistress   
3  6893387fba407998dac8dd2a                       musician   
4  6893387eba407998dac8d9ba  correctional services manager   

                                OCCUPATION_ALTLABELS  \
0  choir leader\nband leader\nchoir director\nmus...   
1  orchestra conductor\nmusic conductor\nband dir...   
2  choirmistress\nchoir director\nchoral director...   
3  musician\noboe player\nmusic player\nmandolin ...   
4  chief prison officer\ncorrectional institution...   

                              OCCUPATION_DESCRIPTION           SECTOR  \
0  Musical conductors lead ensembles of musicians...  Tourism and art   
1  Music directors lead musical groups such as or...  Tourism and art   
2  Choirmasters/choirmistresses manage various as...  Tourism and art  